In [1]:
import os
import shutil
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

for folder in [folder_stream, folder_output, folder_checkpoint]:
    shutil.rmtree(folder, ignore_errors=True)

os.makedirs(folder_stream, exist_ok=True)
print("Persiapan selesai. Folder siap digunakan")

26/11/11 22:53:21 WARN Utils: Your hostname, nabill-IdeaPad-3-14IML05 resolves to a loopback address: 127.0.1.1; using 192.168.11.111 instead (on interface wlp0s20f3)
26/11/11 22:53:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 22:53:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Persiapan selesai. Folder siap digunakan


In [2]:
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

df_stream = spark.readStream.schema(skema_pesanan).option("header", "true").csv(folder_stream)

In [3]:
df_dengan_total = df_stream.withColumn("total_harga", F.col("qty") * F.col("harga_satuan"))

df_agregasi_cabang = df_dengan_total.groupBy("nama_cabang").agg(
    F.sum("total_harga").alias("total_pendapatan"),
    F.count("pesanan_id").alias("jumlah_pesanan"))

query1 = df_agregasi_cabang.writeStream.queryName("agregasi_cabang").outputMode("complete").format("memory").start()

In [6]:
df_stream2 = spark.readStream.schema(skema_pesanan).option("header", "true").csv(folder_stream)
df_dengan_total2 = df_stream2.withColumn("total_harga", F.col("qty") * F.col("harga_satuan"))

df_agregasi_waktu = df_dengan_total2.groupBy(
    F.window(F.col("waktu_pesan"), "2 minutes")).agg(
    F.sum("total_harga").alias("total_pendapatan_window"))

query2 = df_agregasi_waktu.writeStream.queryName("agregasi_waktu").outputMode("complete").format("memory").start()

In [10]:
with open(os.path.join(folder_stream, "batch1.csv"), "w") as f:
    f.write("pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n")
    f.write("P001,Jakarta,Kopi Susu,2,20000,2026-11-11 10:00:15\n")
    f.write("P002,Bandung,Americano,1,18000,2026-11-11 10:00:45\n")

time.sleep(5)
print("=== Hasil Stelah Batch 1 ===")
spark.sql("SELECT * FROM agregasi_cabang").show()


with open(os.path.join(folder_stream, "batch2.csv"), "w") as f:
    f.write("pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n")
    f.write("P003,Jakarta,Latte,3,25000,2026-11-11 10:01:20\n")
    f.write("P004,Surabaya,Matcha,2,22000,2026-11-11 10:01:50\n")

time.sleep(5)
print("=== Hasil Stelah Batch 2 ===")
spark.sql("SELECT * FROM agregasi_cabang").show()


with open(os.path.join(folder_stream, "batch3.csv"), "w") as f:
    f.write("pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n")
    f.write("P005,Bandung,Kopi Susu,4,20000,2026-11-11 10:02:10\n")
    f.write("P006,Surabaya,Espresso,1,15000,2026-11-11 10:02:40\n")

time.sleep(5)
print("=== Hasil Stelah Batch 3 ===")
spark.sql("SELECT * FROM agregasi_cabang").show()

print("=== hasil Agregasi Window Waktu ===")
spark.sql("SELECT * FROM agregasi_waktu").show(truncate=False)

=== Hasil Stelah Batch 1 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           98000|             2|
|    Jakarta|          115000|             2|
|   Surabaya|           59000|             2|
+-----------+----------------+--------------+

=== Hasil Stelah Batch 2 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           98000|             2|
|    Jakarta|          115000|             2|
|   Surabaya|           59000|             2|
+-----------+----------------+--------------+

=== Hasil Stelah Batch 3 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|    Bandung|           98000|             2|
|    Jakarta|          115000|             2|
|   Surabaya|           59000|       

In [12]:
query3 = df_stream.writeStream.format("parquet").option("path", folder_output).option("checkpointLocation", folder_checkpoint).outputMode("append").start()

with open(os.path.join(folder_stream, "batch4.csv"), "w") as f:
    f.write("pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan\n")
    f.write("P007,Jakarta,Cappuccino,2,24000,2026-11-11 10:03:10\n")
    f.write("P008,Bandung,Mocha,1,26000,2026-11-11 10:03:30\n")

time.sleep(6)

print("=== Daftar File di tugas10_output: ===", os.listdir(folder_output))
spark.read.parquet(folder_output).show(truncate=False)

=== Daftar File di tugas10_output: === ['.part-00002-735dc847-9d96-4b68-ad83-6c9de285595b-c000.snappy.parquet.crc', 'part-00003-0a6fff88-c435-4681-8e9b-0bd86efef3cb-c000.snappy.parquet', 'part-00000-22d18023-4533-4336-bdd4-f6278b40a51f-c000.snappy.parquet', 'part-00002-735dc847-9d96-4b68-ad83-6c9de285595b-c000.snappy.parquet', '.part-00003-0a6fff88-c435-4681-8e9b-0bd86efef3cb-c000.snappy.parquet.crc', '_spark_metadata', '.part-00000-22d18023-4533-4336-bdd4-f6278b40a51f-c000.snappy.parquet.crc', 'part-00001-8589929a-c6b0-4e18-9f9f-e801b795d765-c000.snappy.parquet', '.part-00001-8589929a-c6b0-4e18-9f9f-e801b795d765-c000.snappy.parquet.crc']
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P001      |Jakarta    |Kopi Susu |2  |20000       |2026-11-11 10:00:15|
|P002      |Bandung    |Americano |1  |18000       |2026-11-1

In [14]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()